# Estimarea Pretului pentru Procesoare

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from IPython.display import Markdown

plt.style.use("ggplot")
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

Pentru [df_r23](../data/CPU_r23_v2.csv) si [df_bench](../data/CPU_benchmark_v4.csv) datele au fost descarcate din: [CPU Benchmarks Compilation](https://www.kaggle.com/datasets/alanjo/cpu-benchmarks)

In [ ]:
df_r23 = pd.read_csv(r"..\data\CPU_r23_v2.csv")
df_bench = pd.read_csv(r"..\data\CPU_benchmark_v4.csv")

# df.head(20)
# df.isna().sum()
# df.shape
# df.dtypes
# df.describe()

# df["TDP"].value_counts().head(10).plot(kind="barh")
# df["price"].plot(kind="hist", bins=100)

# df["price"].plot(kind="kde") 
# sns.pairplot(df, vars=[""])

# df.plot(kind="scatter", x="TDP", y="price", alpha=0.7)

# Curatarea datelor
Setul de date descarcat din [CPU Benchmarks Compilation](https://www.kaggle.com/datasets/alanjo/cpu-benchmarks) contin date despre aceleas procesoare, doar ca nu sunt legate direct. Pentru a lega seturile de date, functiile:
-    **def clean_r23(row)**: Scoate "ryzen" din cpuName si scoate `-` atunci cand este necesar.
-    **def clean_bench(name)**: Sterge tot ce urmeaza dupa `@` incat adesea este inclus frecventa dupa nume.

In continuare, datele sunt legate prin crearea unui camp comun numit `match_name`. Dupa legare, stergem datele de care nu avem nevoie (cum ar fi testDate) din datele finale.

In [ ]:
def clean_r23(row):
    name = str(row['manufacturer']).lower() + " " + str(row['cpuName']).lower()
    name = name.replace("amd threadripper", "amd ryzen threadripper")
    name = re.sub(r'(core i\d) ', r'\1-', name)
    return name.strip()

def clean_bench(name):
    name = str(name).lower()
    name = name.split('@')[0]
    return name.strip()

df_r23['match_name'] = df_r23.apply(clean_r23, axis=1)
df_bench['match_name'] = df_bench['cpuName'].apply(clean_bench)

df = pd.merge(df_r23, df_bench, on='match_name', how='inner', suffixes=('_r23', '_bench'))
df = df[[
         'manufacturer', 
        #  'cpuName_bench', 
        #  'cpuName_r23', 
         'price', 
         'singleScore', 
         'multiScore', 
         'threads', 
         'baseClock', 
         'turboClock', 
         'type', 
        #  'cpuMark', 
        #  'threadMark',
         'TDP', 
        #  'cores_bench', 
        #  'socket',
         'category'
        #  'cores_r23', e lafel ca cel din benchmark, asa ca luam doar unul
        #  'match_name',
        #  'cpuValue', derivat din mark / price
        #  'threadValue', derivat din threadmark / price
        #  'powerPerf', derivat din cpu mark / tdp
        #  'testDate', data testari care este irelevanta
         ]].copy()

df['manufacturer'] = df['cpuName'].apply(lambda x: str(x).split()[0] if pd.notnull(x) else 'Unknown')

print("Unique manufacturers found:")
print(df['manufacturer'].unique())

df = df_bench
df = df[[
    'cpuName', 
    'price', 
    'cpuMark', 
    # 'cpuValue', 
    'threadMark', 
    # 'threadValue',
    'TDP', 
    # 'powerPerf', 
    'cores', 
    # 'testDate', 
    # 'socket', 
    'category',
    # 'match_name'
    ]].copy()
# df.head()
# df.dtypes


# plt.figure(figsize=(10, 6))
# sns.scatterplot(data=df, x='price', y='multiScore', hue='manufacturer', style='category', alpha=0.7)
# plt.title('CPU Value: Price vs. Multi-core Performance (cpuMark)')
# plt.xscale('log') # Log scale is often better for price since it ranges from $50 to $7000+
# plt.show()

# plt.figure(figsize=(10, 6))
# sns.barplot(data=df, x='TDP', y='multiScore', hue='manufacturer', alpha=0.7)
# plt.title('Power Efficiency: TDP vs. Performance (sized by core count)')
# plt.show()

Index(['cpuName', 'price', 'cpuMark', 'cpuValue', 'threadMark', 'threadValue',
       'TDP', 'powerPerf', 'cores', 'testDate', 'socket', 'category',
       'match_name'],
      dtype='str')

# Analiza exploratorie
Setul de date contine in total

In [11]:
print(df.shape)
print(df.dropna(subset=['price']).shape)

(206, 10)
(167, 10)
